# HuBERT + CTC fine-tuned for Icelandic (our training)

Course baseline family: **HuBERT + CTC head, fine-tuned for your language**. No published Icelandic HuBERT ASR checkpoint was found, so we fine-tune one ourselves.

- Base encoder: `utter-project/mHuBERT-147` (multilingual HuBERT base, 95M params, **no ASR head**)
- We add a CTC head over the train-only character vocabulary (`output/ctc_vocab.json`) and fine-tune on the frozen v1 **train** split (15 h).
- **Dev** picks the best epoch (lowest dev WER, early stopping). **Test** is decoded once at the end with that checkpoint.
- All settings live in `Config` in `training/train_hubert_ctc.py`; the seed is 2105.

This notebook runs top to bottom with no manual steps, so on Kaggle you can use **Save Version → Save & Run All (Commit)** and close the tab; outputs appear in the version's Output tab.
Kaggle settings: Accelerator **GPU T4 x2**, Internet **On**.

In [ ]:
!nvidia-smi

## 1. Repository and audio (train + dev + test)

In [ ]:
import os
WORK = "/kaggle/working" if "KAGGLE_KERNEL_RUN_TYPE" in os.environ else "/content"   # Colab also has a /kaggle folder
REPO_DIR = f"{WORK}/icelandic-asr-project3"

# Fixed commits, so a rerun uses exactly the data and code behind the reported results.
DATA_REPO = "https://github.com/RanimAlyaseen/icelandic-asr-project3.git"
DATA_COMMIT = "ef4d5156cb599348dd9dc55934ff391cc8c5ee5d"   # frozen v1 audio, manifests and references
CODE_REPO = "https://github.com/MuhammetGoksenin/icelandic-asr-project3.git"
CODE_COMMIT = "d830083bb3c7334b64a1bf8c256b2dac7206db8b"                             # baselines/, training/, tests, requirements-modeling.txt
CODE_PATHS = "baselines training tests/test_pretrained.py tests/test_hubert_ctc.py requirements-modeling.txt"
assert len(CODE_COMMIT) == 40, "pin CODE_COMMIT to a full commit hash"

# Kaggle images ship without git-lfs
!which git-lfs >/dev/null || (apt-get -qq update && apt-get -qq install -y git-lfs)
!git lfs install
if not os.path.exists(REPO_DIR):
    !GIT_LFS_SKIP_SMUDGE=1 git clone -q {DATA_REPO} {REPO_DIR}
%cd {REPO_DIR}
!GIT_LFS_SKIP_SMUDGE=1 git checkout -q {DATA_COMMIT}
!git lfs pull --include="data/cleaned/samromur-21.05-20h-v1/**" --exclude=""
!GIT_LFS_SKIP_SMUDGE=1 git fetch -q {CODE_REPO} {CODE_COMMIT}
!git checkout {CODE_COMMIT} -- {CODE_PATHS}
!echo "Data commit: $(git rev-parse HEAD)"; echo "Code commit: {CODE_COMMIT}"

## 2. Check the code is present


In [ ]:
for path in ["training/train_hubert_ctc.py", "baselines/run_pretrained.py", "tests/test_hubert_ctc.py"]:
    assert os.path.exists(path), f"missing {path}: pull the branch that contains it"

## 3. Pinned packages and tests

In [ ]:
!pip install -q -r requirements-modeling.txt   # pinned modeling environment; keeps Kaggle's torch 2.11.0+cu130
# discover -s tests: Kaggle has an unrelated 'tests' package in site-packages
!python -m unittest discover -s tests -p "test_hubert_ctc.py" -v
!python -m unittest discover -s tests -p "test_pretrained.py" -v

## 4. Smoke test (64 train clips, 1 epoch) — checks the whole pipeline in a few minutes

In [ ]:
!python training/train_hubert_ctc.py --train --limit 64 --epochs 1
!rm -rf models/hubert_ctc results/hubert_ctc

## 5. Full training
Expect roughly 3–5 hours on Kaggle T4. If the session stops, rerun cells 1–3 and then `--train --resume` (checkpoints are in `models/hubert_ctc/checkpoints`; on Kaggle they survive only in a committed version's output).

In [ ]:
!python training/train_hubert_ctc.py --train

In [ ]:
import json
from IPython.display import Image, display
print(json.dumps(json.load(open("results/hubert_ctc/training_summary.json")), indent=2)[:1500])
display(Image("results/hubert_ctc/training_curves.png"))

## 6. Test — once, with the dev-selected checkpoint

In [ ]:
!python training/train_hubert_ctc.py --evaluate-test

## 7. Save
- `results/hubert_ctc/` goes into the repository (commit it from **your own** account).
- `models/hubert_ctc/best` is ~380 MB. Publish it on the Hugging Face Hub (optional cell below; add `HF_TOKEN` under Add-ons → Secrets) and write its link and revision in the README.

In [ ]:
!pip freeze > results/hubert_ctc/pip_freeze.txt
!zip -qr {WORK}/hubert_ctc_results.zip results/hubert_ctc
print(f"Download {WORK}/hubert_ctc_results.zip from the Output panel.")

In [ ]:
# Optional: publish the fine-tuned weights
PUBLISH = False
HUB_REPO = "MuhammetGoksenin/mhubert-147-icelandic-ctc-samromur-v1"   # change if you like
if PUBLISH:
    from kaggle_secrets import UserSecretsClient
    from huggingface_hub import HfApi, login
    login(UserSecretsClient().get_secret("HF_TOKEN"))
    HfApi().create_repo(HUB_REPO, exist_ok=True)
    info = HfApi().upload_folder(repo_id=HUB_REPO, folder_path="models/hubert_ctc/best")
    print("Published:", HUB_REPO, info.oid)